# R5 session
One Qiskit version per session. Settings: **CPU, Internet OFF**, pinned environment. Inputs: the `r5-code` dataset, the `r5-deps` dataset and the `r5-base-<version>` dataset of this session.

Edit only the `BASE` line. If the session is interrupted, run it again: finished stages and events are skipped.

In [ ]:
BASE = "2.4.2"

In [ ]:
import glob, json, os, subprocess, sys, time
code = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/r5/session5.py", recursive=True)[0]))
builds = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/faults", recursive=True)
          if glob.glob(os.path.join(os.path.dirname(p), "wheelhouse", f"qiskit-{BASE}-*.whl"))][0]
deps = os.path.dirname(glob.glob("/kaggle/input/**/z3_solver-*.whl", recursive=True)[0])
work = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
site, tmp, out = f"{work}/site_{BASE}", f"{work}/r5tmp_{BASE}", f"/kaggle/working/out_{BASE}"
print("code:", code, "| builds:", builds, "| deps:", deps, "| work:", work)
print(subprocess.run(["df", "-h", work], capture_output=True, text=True).stdout, os.cpu_count(), "cores", sys.version)

In [ ]:
# clean Qiskit of this version and its pinned dependencies, offline
if not os.path.exists(f"{site}/qiskit"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", f"{builds}/wheelhouse",
                        "--find-links", deps, "--target", site, "-r", f"{builds}/wheelhouse/requirements_exact.txt"],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-1500:])
v = subprocess.run([sys.executable, "-c", "import qiskit; print(qiskit.__version__, qiskit.__file__)"],
                   env=dict(os.environ, PYTHONPATH=site), capture_output=True, text=True)
print(v.stdout, v.stderr[-500:])
assert v.stdout.startswith(BASE)

In [ ]:
# the session (hours). Its log is printed at the end and kept in out_<version>/session.log
cp = f"{code}:{code}/cart/cart/src:{code}/cart/obsaware/src"
env = dict(os.environ, PYTHONPATH=f"{site}:{cp}", R5_CODE_PATH=cp, REVEAL_DATA=f"{code}/data", PYTHONUNBUFFERED="1")
os.makedirs(out, exist_ok=True)
with open(f"{out}/session.log", "a") as log:
    p = subprocess.run([sys.executable, "-m", "r5.session5", "--base", BASE, "--site", site, "--builds", builds,
                        "--out", out, "--tmp", tmp], cwd=code, env=env, stdout=log, stderr=subprocess.STDOUT)
print("exit code", p.returncode)
print("".join(open(f"{out}/session.log").readlines()[-25:]))

In [ ]:
# summary and one zip of the outputs to download
import shutil
print("DONE" if os.path.exists(f"{out}/DONE") else "NOT DONE", "| events:", len(glob.glob(f"{out}/events/*.json")))
b = [json.loads(l)["median_s"] for l in open(f"{out}/benchmark.jsonl")]
print("benchmark drift from first reading: max", f"{max(abs(x / b[0] - 1) for x in b):.1%}")
shutil.make_archive(f"/kaggle/working/out_{BASE}", "zip", out)
print("zip:", f"/kaggle/working/out_{BASE}.zip")